## Loading the data — part 1: statistics

This notebook uses four files, split across two folders in the repo, so they're uploaded in two steps. First, upload the three Eurostat CSVs from `Geospacific/data/processed/`: household internet access, individual internet use, and digital skills. Hold Ctrl (Cmd on Mac) in the file dialog to select all three at once.

In [ ]:
from google.colab import files

uploaded_csv = files.upload()
# Select all three from data/processed/:
# - household_internet_access.csv
# - individual_internet_use.csv
# - digital_skills_by_age.csv

print("Uploaded:", list(uploaded_csv.keys()))

## Loading the data — part 2: country boundaries

Next, upload the country boundary map. This file lives in a different folder, `Geospacific/data/geo/`, so it needs a separate upload step: `europe_countries_boundaries.geojson` (European country boundaries from Eurostat/GISCO, supplemented with Kosovo from OpenStreetMap — see the project README for details). **Note:** this file was corrected this session (Kosovo's boundary ring direction was flipped to fix a rendering bug) — re-upload a fresh copy if you have an older one saved locally.

In [ ]:
uploaded_geo = files.upload()
# Select from data/geo/:
# - europe_countries_boundaries.geojson

print("Uploaded:", list(uploaded_geo.keys()))

## Joining the statistics to the map

Statistics and country boundaries are joined on country code (`geo_code` in the data, `CNTR_ID` on the map). The check below shows which records in the data have no matching shape on the map — expected to be EU/euro-area aggregate values, which have no boundary of their own.

In [ ]:
import pandas as pd
import json

df = pd.read_csv('household_internet_access.csv')

with open('europe_countries_boundaries.geojson') as f:
    geojson = json.load(f)

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
csv_ids = set(df['geo_code'])
print("In CSV but not on the map:", csv_ids - geo_ids)  # expected: EU aggregates (EU27_2020, etc.)

## Filling gaps in the data

A country is shown in amber before the first year with available data. If a later year is missing, the value from the last available year is used instead.

In [ ]:
# Build a full country x year grid, then forward-fill missing years with
# each country's last known value, so no country disappears mid-animation.

# Keep only actual countries, not EU/EA aggregates
df_countries = df[df['geo_code'].isin(geo_ids)].sort_values('year')

all_years = range(df_countries['year'].min(), df_countries['year'].max() + 1)
all_countries = df_countries[['geo_code', 'geo_label']].drop_duplicates()

full_index = pd.MultiIndex.from_product(
    [all_countries['geo_code'], all_years],
    names=['geo_code', 'year']
)

df_filled = (
    df_countries.set_index(['geo_code', 'year'])
    .reindex(full_index)
    .reset_index()
)

# geo_label gets lost by reindex (it's not part of the index) - restore it
df_filled['geo_label'] = df_filled['geo_code'].map(
    all_countries.set_index('geo_code')['geo_label']
)

# Forward-fill the value within each country, ordered by year. Years before
# a country's first real data point stay NaN - intentionally not dropped,
# so the map below can show them as "no data" rather than omitting them.
df_filled = df_filled.sort_values(['geo_code', 'year'])
df_filled['pct_households_with_internet'] = (
    df_filled.groupby('geo_code')['pct_households_with_internet'].ffill()
)

n_missing = df_filled['pct_households_with_internet'].isna().sum()
print(f"Total rows: {len(df_filled)}, still missing (no data yet that year): {n_missing}")

## Highlighting missing data on the map

Rather than dropping early years or letting a missing country blend in with a pale (low-value) color on the Blues scale - which is exactly what made Sweden's 2002-2004 gap look like "0% internet access" earlier - countries with no data yet are colored a distinct amber, both on the map and in the "not yet reporting" label below it, so the two are visually tied together at a glance.

In [ ]:
# Shared color for "no data yet" - used both for the map's missing-country
# fill and the text label below it, so the two read as the same thing.
MISSING_COLOR = 'rgb(246,217,168)'

# For each year, which tracked countries have no data yet? (NaN after
# forward-filling means genuinely nothing reported up to and including
# that year, not just a gap that got carried forward.)
missing_by_year = {}
for year in sorted(df_filled['year'].unique()):
    year_rows = df_filled[df_filled['year'] == year]
    missing = sorted(year_rows.loc[year_rows['pct_households_with_internet'].isna(), 'geo_code'])
    missing_by_year[int(year)] = missing

for year, missing in missing_by_year.items():
    label = ", ".join(missing) if missing else "none - full coverage"
    print(f"{year}: not yet reporting - {label}")

## Map over time

An interactive map of Europe with a year slider (2002–2025) — color shows the percentage of households with internet access. Countries with no data yet that year are shown in amber, matching the "not yet reporting" label below the map. Play the slider as an animation, or step through individual years and hover over a country to see the exact value.

The map is sized larger here (1000x700) so it's easier to read the country shapes and labels. Country codes in the "not yet reporting" label can be looked up in the legend table right after the map.

In [ ]:
import plotly.graph_objects as go

# Ocean and "world we don't track" get their own colors, distinct from
# MISSING_COLOR - otherwise Kaliningrad, Russia, Belarus etc. would look
# exactly like a tracked country with missing data.
OCEAN_COLOR = '#DFF2E8'
UNTRACKED_COLOR = '#E7E5DB'
name_lookup = all_countries.set_index('geo_code')['geo_label']

# --- Why Kosovo gets its own pair of traces -------------------------------
# Kosovo's shape sits almost entirely inside Serbia's: GISCO draws Serbia
# without a hole cut out for Kosovo (it doesn't recognize its independence),
# so the two overlap by ~99% of Kosovo's area. Two separate Plotly rules
# then work against each other within a single trace:
#   * FILL   - countries are drawn in list order, so the LAST one wins and
#              covers the ones before it.
#   * HOVER  - Plotly walks the same list and stops at the FIRST country
#              whose shape contains the cursor.
# So inside one trace, whichever of the two is listed last is visible but
# un-hoverable, and whichever is first is hoverable but hidden - there's no
# ordering that fixes both. Putting Kosovo in its own trace, added after the
# others, sidesteps it entirely: later traces both draw on top AND take
# hover priority, so Kosovo ends up correct on both counts.
MAIN_COUNTRIES = sorted(c for c in all_countries['geo_code'] if c != 'XK')

value_lookup = df_filled.set_index(['geo_code', 'year'])['pct_households_with_internet']
years = sorted(df_filled['year'].unique())


def country_state(code, year, missing_set):
    """Returns (data_z, data_hover, amber_z, amber_hover) for one country.

    A country is either coloured by its value or flagged amber as "no data
    yet" - never both, so the unused one gets None/"" that year."""
    name = name_lookup.get(code, code)
    if code in missing_set:
        return None, "", 1, f"{name}<br>No data yet"
    v = value_lookup.get((code, year))
    return v, f"{name}<br>{v:.1f}%", None, ""


def frame_data(year):
    """Builds z/hover arrays for all four traces for a given year."""
    missing_set = set(missing_by_year.get(year, []))
    z_data, hover_data, z_amber, hover_amber = [], [], [], []
    for c in MAIN_COUNTRIES:
        dz, dh, az, ah = country_state(c, year, missing_set)
        z_data.append(dz); hover_data.append(dh)
        z_amber.append(az); hover_amber.append(ah)
    xk_dz, xk_dh, xk_az, xk_ah = country_state('XK', year, missing_set)
    return dict(
        main_data=(z_data, hover_data), main_amber=(z_amber, hover_amber),
        xk_data=([xk_dz], [xk_dh]), xk_amber=([xk_az], [xk_ah])
    )


# Small color-swatch legend for the map's background colors - these aren't
# part of the Blues scale, so they need their own key.
LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes, legend_annotations = [], []
_start_x, _y_pos, _box_w, _gap = 0.0, 1.045, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(
        type='rect', xref='paper', yref='paper',
        x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
        fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)
    ))
    legend_annotations.append(dict(
        text=label, xref='paper', yref='paper',
        x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
        xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)
    ))


def reporting_annotation(year):
    # Always returns exactly ONE annotation, visually blank when every
    # country is reporting. Two separate quirks are being worked around:
    #   1. Frames merge annotations by position rather than replacing the
    #      list, so a frame with fewer annotations than the previous one
    #      leaves the leftover on screen.
    #   2. During Play, an empty-string text doesn't overwrite the previous
    #      frame's text - the last non-blank year stayed visible for the
    #      rest of the animation. A single space counts as a real value and
    #      overwrites properly, while rendering as nothing.
    missing = missing_by_year.get(year, [])
    return dict(
        text=f"Not yet reporting: {', '.join(missing)}" if missing else " ",
        xref='paper', yref='paper', x=0.5, y=-0.06,
        showarrow=False, font=dict(size=11, color=MISSING_COLOR)
    )


def traces_for(year, first=False):
    """Four traces: main data, main amber, Kosovo data, Kosovo amber.
    geojson is only attached on the first build - frames just swap values."""
    d = frame_data(year)
    geo = dict(geojson=geojson, featureidkey='properties.CNTR_ID') if first else {}
    return [
        go.Choropleth(locations=MAIN_COUNTRIES, z=d['main_data'][0], text=d['main_data'][1],
                      hoverinfo='text', colorscale='Blues', zmin=0, zmax=100,
                      marker_line_color='white', marker_line_width=0.5,
                      colorbar=dict(title='% of households<br>with internet'), **geo),
        go.Choropleth(locations=MAIN_COUNTRIES, z=d['main_amber'][0], text=d['main_amber'][1],
                      hoverinfo='text', colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
                      showscale=False, marker_line_color='white', marker_line_width=0.3, **geo),
        go.Choropleth(locations=['XK'], z=d['xk_data'][0], text=d['xk_data'][1],
                      hoverinfo='text', colorscale='Blues', zmin=0, zmax=100, showscale=False,
                      marker_line_color='white', marker_line_width=0.5, **geo),
        go.Choropleth(locations=['XK'], z=d['xk_amber'][0], text=d['xk_amber'][1],
                      hoverinfo='text', colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
                      showscale=False, marker_line_color='white', marker_line_width=0.3, **geo)
    ]


first_year = years[0]

fig = go.Figure(
    data=traces_for(first_year, first=True),
    frames=[
        go.Frame(name=str(year), data=traces_for(year),
                 layout=dict(annotations=legend_annotations + [reporting_annotation(year)]))
        for year in years
    ]
)

# Continental-Europe framing, ignoring overseas territories that would
# otherwise zoom the map out.
fig.update_geos(
    visible=False,
    lonaxis_range=[-25, 45],
    lataxis_range=[33, 72],
    showland=True,
    landcolor=UNTRACKED_COLOR,
    showocean=True,
    oceancolor=OCEAN_COLOR,
    projection_type='equirectangular'
)

fig.update_layout(
    title='Household internet access in Europe, 2002-2025',
    dragmode=False,
    width=1000,
    height=700,
    margin=dict(t=110, b=110),
    shapes=legend_shapes,
    annotations=legend_annotations + [reporting_annotation(first_year)],
    # Slider steps use method='animate' so that pressing Play moves the
    # slider handle and year label along with the map - a slider built on
    # method='update' changes the map but leaves the handle behind.
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix='Year: '),
        x=0.18, len=0.80, y=-0.14, pad=dict(t=0, b=0),
        steps=[
            dict(method='animate', label=str(year),
                 args=[[str(year)], dict(mode='immediate',
                                          frame=dict(duration=0, redraw=True),
                                          transition=dict(duration=0))])
            for year in years
        ]
    )],
    updatemenus=[dict(
        type='buttons', direction='left', showactive=False,
        x=0.0, xanchor='left', y=-0.15, yanchor='top', pad=dict(t=0, r=10),
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, dict(frame=dict(duration=400, redraw=True),
                                   fromcurrent=True, transition=dict(duration=0))]),
            dict(label='Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                     mode='immediate', transition=dict(duration=0))])
        ]
    )]
)

fig.show()

## Country code legend

Reference table for every country code used in the map and in the "not yet reporting" label above, so any code (e.g. `XK`) can be looked up without hovering or guessing.

In [ ]:
legend_df = (
    df_filled[['geo_code', 'geo_label']]
    .drop_duplicates()
    .sort_values('geo_code')
    .reset_index(drop=True)
    .rename(columns={'geo_code': 'Code', 'geo_label': 'Country'})
)
legend_df

## Diagnostics

Sanity checks used while debugging the animation (country coverage per year, and the dtype/ordering of the year column). Kept here for reference — safe to ignore or remove once the map looks right.

In [ ]:
# How many countries have REAL data in each year (excluding the still-amber,
# not-yet-reporting ones)? Should only ever go up, never dip.
counts_per_year = (
    df_filled.dropna(subset=['pct_households_with_internet'])
    .groupby('year')['geo_code'].nunique()
)
print(counts_per_year)

In [ ]:
print(df_filled['year'].dtype)
print(sorted(df_filled['year'].unique()))